In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data_path = "../data/data.csv"
column_names = ["density", "cutting_speed", "feed_rate", "depth", "axial_force", "cutting_force"]
target = "cutting_force"
input_columns = ["density", "cutting_speed", "feed_rate", "depth", "axial_force"]

split_seed = 42
n_splits = 5

In [3]:
df = pd.read_csv(data_path)
df.columns = column_names
print(f"Total rows: {len(df):,}")

X = df[input_columns].values
y = df[target].values

Total rows: 283,140


In [4]:
kf = KFold(n_splits=n_splits, shuffle=True, random_state=split_seed)

fold_results = []

for fold_index, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)
    linear_preds = linear_model.predict(X_test)

    hgb_model = HistGradientBoostingRegressor(random_state=split_seed)
    hgb_model.fit(X_train, y_train)
    hgb_preds = hgb_model.predict(X_test)

    for name, preds in [("LinearRegression", linear_preds), ("HistGradientBoosting", hgb_preds)]:
        mse = mean_squared_error(y_test, preds)
        result = {
            "Fold": fold_index,
            "Model": name,
            "MAE": mean_absolute_error(y_test, preds),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y_test, preds),
        }
        fold_results.append(result)
        print(f"Fold {fold_index}/{n_splits} | {name}: RMSE={result['RMSE']:.4f} | R2={result['R2']:.4f}")

fold_summary = pd.DataFrame(fold_results)

Fold 1/5 | LinearRegression: RMSE=97.3060 | R2=0.7810
Fold 1/5 | HistGradientBoosting: RMSE=6.9435 | R2=0.9989
Fold 2/5 | LinearRegression: RMSE=98.2997 | R2=0.7795
Fold 2/5 | HistGradientBoosting: RMSE=6.7312 | R2=0.9990
Fold 3/5 | LinearRegression: RMSE=97.8030 | R2=0.7794
Fold 3/5 | HistGradientBoosting: RMSE=6.8539 | R2=0.9989
Fold 4/5 | LinearRegression: RMSE=98.0546 | R2=0.7774
Fold 4/5 | HistGradientBoosting: RMSE=6.7101 | R2=0.9990
Fold 5/5 | LinearRegression: RMSE=98.4786 | R2=0.7832
Fold 5/5 | HistGradientBoosting: RMSE=6.9214 | R2=0.9989


In [5]:
print("Per-fold results:")
display(fold_summary.round(4))

print("\nAggregate across folds (mean +/- std):")
summary = fold_summary.groupby("Model")[["MAE", "MSE", "RMSE", "R2"]].agg(["mean", "std"])
display(summary.round(4))

Per-fold results:


,Fold,Model,MAE,MSE,RMSE,R2
0,1,LinearRegression,73.7083,9468.4486,97.3060,0.7810
1,1,HistGradientBoosting,4.9456,48.2125,6.9435,0.9989
2,2,LinearRegression,74.3608,9662.8226,98.2997,0.7795
3,2,HistGradientBoosting,4.8586,45.3084,6.7312,0.9990
4,3,LinearRegression,74.1497,9565.4287,97.8030,0.7794
5,3,HistGradientBoosting,4.9086,46.9760,6.8539,0.9989
6,4,LinearRegression,74.1076,9614.7138,98.0546,0.7774
7,4,HistGradientBoosting,4.9030,45.0256,6.7101,0.9990
8,5,LinearRegression,74.1061,9698.0335,98.4786,0.7832
9,5,HistGradientBoosting,4.9376,47.9060,6.9214,0.9989



Aggregate across folds (mean +/- std):


MAE                MSE              RMSE          \
                         mean     std       mean      std     mean     std   
Model                                                                        
HistGradientBoosting   4.9107  0.0344    46.6857   1.4626   6.8320  0.1072   
LinearRegression      74.0865  0.2362  9601.8894  89.8009  97.9884  0.4587   

                          R2          
                        mean     std  
Model                                 
HistGradientBoosting  0.9989  0.0000  
LinearRegression      0.7801  0.0022